In [1]:
from datasets import Dataset
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
from ragas import evaluate

f:\anaconda3\envs\eval\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import json

with open("rag_eval_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

eval_questions = data["questions"]
answers = data["answers"]
contexts_list = data["contexts"]
ground_truth = data["ground_truth"]

In [3]:
import os
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI
from langchain_core.embeddings import Embeddings
from openai import OpenAI


dataset = Dataset.from_dict(
    {
        "question": eval_questions,
        "answer": answers,
        "contexts": contexts_list,
        "ground_truth": ground_truth
    }
)

from langchain_openai import ChatOpenAI  

load_dotenv()  
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = "https://api.deepseek.com"  
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")




judge_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0,
)


alikey = os.getenv("DASHSCOPE_API_KEY")

# === Ali embedding ===
class TextV4Embeddings(Embeddings):
    def __init__(self, api_key, base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"):
        self.client = OpenAI(api_key=alikey, base_url=base_url)
        self.model = "text-embedding-v4"
    def embed_documents(self, texts): 
        resp = self.client.embeddings.create(model=self.model, input=texts, dimensions=1024)
        return [d.embedding for d in resp.data]
    def embed_query(self, text):
        return self.embed_documents([text])[0]

# === initialize embedding model ===
emb = TextV4Embeddings(api_key=alikey)



result = evaluate(
    dataset=dataset,
    metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
    llm=judge_llm,
    embeddings=emb,
    is_async=False
)

print(result)
# print(result.to_pandas().head())

Evaluating: 100%|██████████| 40/40 [00:37<00:00,  1.05it/s]


{'context_precision': 0.8200, 'context_recall': 0.8000, 'faithfulness': 0.8035, 'answer_relevancy': 0.5956}
